# 04b — Bogotá NO (nitric oxide) network series

The local SISAIRE export (`Data/COLOMBIA/BG_*.csv`) has no NO variable at all (only NO2) --
confirmed by an exhaustive search across the whole `Data/COLOMBIA` directory, not just
Bogotá. This matters a great deal for O3: per the user's own paper (Casallas et al. 2023,
*Air Quality, Atmosphere & Health*, "Surface, satellite ozone variations in Northern South
America during low anthropogenic emission conditions: a machine learning approach"), the
**NO titration effect (NO + O3 -> NO2 + O2) is the single dominant driver of O3
variability** in Bogotá -- their ML precursor-decomposition attributes roughly 50-70% of
the 2020 ozone changes to the drop in NO, far ahead of NO2, PM2.5, radiation or wind.

The user located the actual raw hourly station data used for that paper at
`~/Documents/Postdoc_MSCA-Bridge/USA_Ozone/DATA_ORIGINAL/NO.csv` -- hourly, wide-format
(one column per station), 2013-01-01 through 2022-12-31, for the same 20 Bogotá stations
(Carvajal, Centro de Alto Rendimiento, Guaymaral, Kennedy, San Cristóbal, Suba, Tunal,
Usaquén, plus others) as the SISAIRE export. This notebook extracts a QC'd daily-mean NO
network series from it, using the exact same conventions as `Scripts/common/qc.py`
(Chauvenet's criterion, physical range checks, >=70% completeness -- matching
`MIN_COVERAGE` in notebook 06).

**Coverage note**: this source only runs through 2022-12-31, while the SISAIRE-based
O3 master dataset runs through 2026. NO will therefore be NaN for 2023-2026 -- this
follows the same pattern as the fire-hotspot coverage gap already documented in notebook
04, and SFS/PySR will simply use fewer complete rows when NO is included as a candidate.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import numpy as np
import pandas as pd

from common.qc import chauvenet_outlier_mask, completeness_by_station_year, select_stations_years

RAW_PATH = Path("/path/to/home/Documents/Postdoc_MSCA-Bridge/USA_Ozone/DATA_ORIGINAL/NO.csv")
OUT_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "bogota"

raw = pd.read_csv(RAW_PATH)
raw = raw.rename(columns={raw.columns[0]: "datetime"})
raw["datetime"] = pd.to_datetime(raw["datetime"], format="%d/%m/%Y %H:%M", errors="coerce")
raw = raw.dropna(subset=["datetime"])
print(raw.shape, raw["datetime"].min(), raw["datetime"].max())
station_cols = [c for c in raw.columns if c != "datetime"]
print(f"{len(station_cols)} stations:", station_cols)


(87648, 21) 2013-01-01 00:00:00 2022-12-31 23:00:00
20 stations: ['BOLIVIA', 'BOSA', 'CARVAJAL', 'CENTRO DE ALTO RENDIMIENTO', 'CIUDAD BOLIVAR', 'FONTIBON', 'GUAYMARAL', 'JAZMIN', 'KENNEDY', 'LAS FERIAS', 'MIN AMBIENTE', 'PUENTE ARANDA', 'SAN CRISTOBAL', 'SUBA', 'TUNAL', 'USAQUEN', 'USME', 'COLINA', 'MOVIL FONTIBON', 'MOVIL 7MA']


## Melt to long format, physical range check, Chauvenet outlier flagging (per station, on hourly values)

In [2]:
long = raw.melt(id_vars="datetime", value_vars=station_cols, var_name="station", value_name="no")
long["no"] = pd.to_numeric(long["no"], errors="coerce")
print("raw non-null hourly obs:", long["no"].notna().sum())

# Physical range check -- mirror the NO2 range from common/qc.py's PHYSICAL_RANGES (0-500).
NO_RANGE = (0.0, 500.0)
bad = (long["no"] < NO_RANGE[0]) | (long["no"] > NO_RANGE[1])
print(f"[range check] no: {int(bad.sum())} values outside {NO_RANGE} -> NaN")
long.loc[bad, "no"] = np.nan

mask = long.groupby("station")["no"].transform(lambda s: chauvenet_outlier_mask(s))
print(f"[Chauvenet] no: flagged {int(mask.sum())} hourly outliers")
long.loc[mask, "no"] = np.nan
print("non-null hourly obs after QC:", long["no"].notna().sum())


raw non-null hourly obs: 719167
[range check] no: 120 values outside (0.0, 500.0) -> NaN
[Chauvenet] no: flagged 5416 hourly outliers
non-null hourly obs after QC: 713631


## Hourly -> daily (per station, requires >=18/24 non-null hours), then completeness filtering

In [3]:
long["date"] = long["datetime"].dt.floor("D")
daily_station = (
    long.groupby(["station", "date"])["no"]
    .agg(no_mean="mean", n_hours="count")
    .reset_index()
)
MIN_HOURS = 18  # >=75% of 24, matching the group's completeness convention
daily_station.loc[daily_station["n_hours"] < MIN_HOURS, "no_mean"] = np.nan
daily_station = daily_station.rename(columns={"no_mean": "no"})[["station", "date", "no"]]
print(daily_station.shape, "station-days with a valid daily mean:", daily_station["no"].notna().sum())

completeness = completeness_by_station_year(daily_station, value_col="no")
MIN_COVERAGE = 0.70  # matches notebook 06's MIN_COVERAGE
good_station_years = select_stations_years(completeness, threshold=MIN_COVERAGE)
print(f"{len(good_station_years)} / {len(completeness)} station-years meet >={MIN_COVERAGE:.0%} completeness")
good_station_years[["station", "year", "completeness"]].sort_values("completeness", ascending=False).head(10)


(73040, 3) station-days with a valid daily mean: 29699
75 / 108 station-years meet >=70% completeness


,station,year,completeness
85,SAN CRISTOBAL,2022,0.997260
104,USAQUEN,2022,0.997260
31,FONTIBON,2020,0.994536
63,LAS FERIAS,2019,0.994521
20,CENTRO DE ALTO RENDIMIENTO,2019,0.994521
22,CENTRO DE ALTO RENDIMIENTO,2021,0.991781
54,KENNEDY,2020,0.989071
87,SUBA,2014,0.989041
106,USME,2021,0.989041
17,CENTRO DE ALTO RENDIMIENTO,2016,0.986339


In [4]:
daily_station["year"] = daily_station["date"].dt.year
keep_keys = set(zip(good_station_years["station"], good_station_years["year"]))
daily_station["keep"] = daily_station.apply(lambda r: (r["station"], r["year"]) in keep_keys, axis=1)
qualified = daily_station[daily_station["keep"]]
print("qualifying station-days:", qualified.shape)

network = qualified.groupby("date")["no"].mean().reset_index().rename(columns={"no": "no"})
network = network.sort_values("date").reset_index(drop=True)
print(network.shape, network["date"].min(), network["date"].max())
print("non-null network-day count:", network["no"].notna().sum())
network.head()


qualifying station-days: (27386, 5)
(3652, 2) 2013-01-01 00:00:00 2022-12-31 00:00:00
non-null network-day count: 3650


,date,no
0,2013-01-01,13.303125
1,2013-01-02,28.274830
2,2013-01-03,36.501766
3,2013-01-04,32.878578
4,2013-01-05,36.069792


## Save

In [5]:
out_path = OUT_DIR / "bogota_no_daily.parquet"
network.to_parquet(out_path, index=False)
print("Saved:", out_path, network.shape)


Saved: /path/to/project/Data/processed/colombia/bogota/bogota_no_daily.parquet (3652, 2)
